**Retail Sales ETL Pipeline**

This project demonstrates a simple ETL pipeline using Python and SQL Server.

**Objectives**

- Load a retail sales dataset from a CSV file
- Perform exploratory data analysis
- Clean and prepare the dataset
- Transform the data into a dimensional star schema
- Load the data warehouse tables in SQL Server
- Prepare optimized views for BI reporting

**Libraries Used**

- pandas → data manipulation and transformation
- numpy → numerical operations
- pyodbc → SQL Server connectivity
- sqlalchemy → database engine used for loading data with pandas

In [204]:
import pandas as pd
import numpy as np
import pyodbc
from sqlalchemy import create_engine

**Load Source Dataset**

The source dataset is a CSV file containing supermarket transaction records.

Each row represents a single sales transaction and includes customer details,
product information, payment method, and sales metrics.

The dataset will be loaded into a pandas dataframe for further processing.

In [205]:
df = pd.read_csv('../data/supermarket_sales.csv')

df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3


**Exploratory Data Analysis**

Before transforming the dataset, we perform basic exploration to understand the structure of data:

- Column data types
- Missing values
- Reviewing numerical statistics
- Understanding dataset size and structure

In [206]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   str    
 1   Branch                   1000 non-null   str    
 2   City                     1000 non-null   str    
 3   Customer type            1000 non-null   str    
 4   Gender                   1000 non-null   str    
 5   Product line             1000 non-null   str    
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Sales                    1000 non-null   float64
 10  Date                     1000 non-null   str    
 11  Time                     1000 non-null   str    
 12  Payment                  1000 non-null   str    
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  1000 non-nu

In [207]:
df.describe()

,Unit price,Quantity,Tax 5%,Sales,cogs,gross margin percentage,gross income,Rating
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.00000
mean,55.672130,5.510000,15.379369,322.966749,307.58738,4.761905,15.379369,6.97270
std,26.494628,2.923431,11.708825,245.885335,234.17651,0.000000,11.708825,1.71858
min,10.080000,1.000000,0.508500,10.678500,10.17000,4.761905,0.508500,4.00000
25%,32.875000,3.000000,5.924875,124.422375,118.49750,4.761905,5.924875,5.50000
50%,55.230000,5.000000,12.088000,253.848000,241.76000,4.761905,12.088000,7.00000
75%,77.935000,8.000000,22.445250,471.350250,448.90500,4.761905,22.445250,8.50000
max,99.960000,10.000000,49.650000,1042.650000,993.00000,4.761905,49.650000,10.00000


In [208]:
df.isnull().sum()

Invoice ID                 0
Branch                     0
City                       0
Customer type              0
Gender                     0
Product line               0
Unit price                 0
Quantity                   0
Tax 5%                     0
Sales                      0
Date                       0
Time                       0
Payment                    0
cogs                       0
gross margin percentage    0
gross income               0
Rating                     0
dtype: int64

**Data Cleaning and Preparation**

Before building the dimensional model, the dataset must be cleaned and standardized.

The following steps are performed:

1. Standardize column names by removing spaces
2. Convert date columns to datetime format
3. Remove duplicate records
4. Verify correct data types

In [209]:
df.columns = df.columns.str.replace(' ', '_')
df.columns

Index(['Invoice_ID', 'Branch', 'City', 'Customer_type', 'Gender',
       'Product_line', 'Unit_price', 'Quantity', 'Tax_5%', 'Sales', 'Date',
       'Time', 'Payment', 'cogs', 'gross_margin_percentage', 'gross_income',
       'Rating'],
      dtype='str')

In [210]:
df['Date'] = pd.to_datetime(df['Date'])

In [211]:
df = df.drop_duplicates()

In [212]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Invoice_ID               1000 non-null   str           
 1   Branch                   1000 non-null   str           
 2   City                     1000 non-null   str           
 3   Customer_type            1000 non-null   str           
 4   Gender                   1000 non-null   str           
 5   Product_line             1000 non-null   str           
 6   Unit_price               1000 non-null   float64       
 7   Quantity                 1000 non-null   int64         
 8   Tax_5%                   1000 non-null   float64       
 9   Sales                    1000 non-null   float64       
 10  Date                     1000 non-null   datetime64[us]
 11  Time                     1000 non-null   str           
 12  Payment                  1000 non-null   str  

**Creating Dimension Tables**

For analytical reporting we transform original transactional dataset into a dimensional model (star-schema).

The following dimensions will be created:

- DimCustomer → customer type and gender
- DimProduct → product category and price
- DimStore → store branch and city
- DimPayment → payment method
- DimDate → date granularity

**Customer Dimension**

In [213]:
dim_customer = df[['Customer_type','Gender']].drop_duplicates()
dim_customer.reset_index(drop=True, inplace=True)
dim_customer['CustomerKey'] = dim_customer.index + 1

dim_customer

,Customer_type,Gender,CustomerKey
0,Member,Female,1
1,Normal,Female,2
2,Member,Male,3
3,Normal,Male,4


**Store Dimension**

In [214]:
dim_store = df[['Branch','City']].drop_duplicates()
dim_store.reset_index(drop=True, inplace=True)
dim_store['StoreKey'] = dim_store.index + 1

dim_store

,Branch,City,StoreKey
0,Alex,Yangon,1
1,Giza,Naypyitaw,2
2,Cairo,Mandalay,3


**Payment Dimension**

In [215]:
dim_payment = df[['Payment']].drop_duplicates()
dim_payment.reset_index(drop=True, inplace=True)
dim_payment['PaymentKey'] = dim_payment.index + 1

dim_payment

,Payment,PaymentKey
0,Ewallet,1
1,Cash,2
2,Credit card,3


**Product Dimension**

In [216]:
dim_product = df[['Product_line','Unit_price']].drop_duplicates()
dim_product.reset_index(drop=True, inplace=True)
dim_product['ProductKey'] = dim_product.index + 1

dim_product

,Product_line,Unit_price,ProductKey
0,Health and beauty,74.69,1
1,Electronic accessories,15.28,2
2,Home and lifestyle,46.33,3
3,Health and beauty,58.22,4
4,Sports and travel,86.31,5
...,...,...,...
988,Health and beauty,40.35,989
989,Home and lifestyle,97.38,990
990,Food and beverages,31.84,991
991,Home and lifestyle,65.82,992


**Date Dimension**

In [217]:
dim_date = pd.DataFrame()

dim_date['Date'] = df['Date'].drop_duplicates().reset_index(drop=True)

dim_date['DateKey'] = dim_date['Date'].dt.strftime('%Y%m%d').astype(int)
dim_date['Year'] = dim_date['Date'].dt.year
dim_date['Month'] = dim_date['Date'].dt.month
dim_date['Day'] = dim_date['Date'].dt.day
dim_date['DayName'] = dim_date['Date'].dt.day_name()

dim_date = dim_date[['DateKey','Date','Year','Month','Day','DayName']]

dim_date

,DateKey,Date,Year,Month,Day,DayName
0,20190105,2019-01-05,2019,1,5,Saturday
1,20190308,2019-03-08,2019,3,8,Friday
2,20190303,2019-03-03,2019,3,3,Sunday
3,20190127,2019-01-27,2019,1,27,Sunday
4,20190208,2019-02-08,2019,2,8,Friday
...,...,...,...,...,...,...
84,20190226,2019-02-26,2019,2,26,Tuesday
85,20190317,2019-03-17,2019,3,17,Sunday
86,20190314,2019-03-14,2019,3,14,Thursday
87,20190204,2019-02-04,2019,2,4,Monday


**Building the Fact Table**

To build the dimensional fact table we must dimensionize key columns from each dimension.

In [218]:
fact = df.merge(dim_store, on=['Branch','City'])
fact = fact.merge(dim_customer, on=['Customer_type','Gender'])
fact = fact.merge(dim_payment, on='Payment')
fact = fact.merge(dim_product, on=['Product_line','Unit_price'])
fact = fact.merge(dim_date[['Date','DateKey']], on='Date')

**FactSales Table**

In [219]:
fact_sales = fact[[
    'Invoice_ID',
    'DateKey',
    'CustomerKey',
    'ProductKey',
    'StoreKey',
    'PaymentKey',
    'Quantity',
    'Unit_price',
    'Tax_5%',
    'Sales',
    'cogs',
    'gross_margin_percentage',
    'gross_income',
    'Rating'
]].copy()

In [220]:
fact_sales['DateKey'] = fact_sales['DateKey'].astype(int)
fact_sales['CustomerKey'] = fact_sales['CustomerKey'].astype(int)
fact_sales['ProductKey'] = fact_sales['ProductKey'].astype(int)
fact_sales['StoreKey'] = fact_sales['StoreKey'].astype(int)
fact_sales['PaymentKey'] = fact_sales['PaymentKey'].astype(int)

numeric_cols = ['Unit_price','Tax_5%','Sales','cogs','gross_margin_percentage','gross_income','Rating']
fact_sales[numeric_cols] = fact_sales[numeric_cols].astype(float)

**Connect to SQL Server**

The transformed dimensional model is loaded into SQL Server using pandas `to_sql()` through a SQLAlchemy engine.

In [221]:
engine = create_engine(
"mssql+pyodbc://@localhost/RetailDW?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

print("Engine Created Successfully!")

Engine Created Successfully!


**Data Quality Checks**

Before loading the transformed data into the data warehouse, basic validation checks are performed to ensure data consistency.

The checks include:

- Verifying row counts
- Confirming primary keys exist
- Ensuring fact table foreign keys align with dimension tables

In [222]:
print("Customer rows:", len(dim_customer))
print("Store rows:", len(dim_store))
print("Product rows:", len(dim_product))
print("Payment rows:", len(dim_payment))
print("Date rows:", len(dim_date))
print("Fact rows:", len(fact_sales))

Customer rows: 4
Store rows: 3
Product rows: 993
Payment rows: 3
Date rows: 89
Fact rows: 1000


**Load Dimension Tables into SQL Server**

After creating the dimension dataframes in Python, we load them
into the SQL Server data warehouse tables.

Each dataframe will be inserted into its corresponding dimension table.

In [228]:
print("Loading dimension tables...")

dim_customer.to_sql('DimCustomer', engine, if_exists='append', index=False)
print("DimCustomer loaded")

dim_store.to_sql('DimStore', engine, if_exists='append', index=False)
print("DimStore loaded")

dim_payment.to_sql('DimPayment', engine, if_exists='append', index=False)
print("DimPayment loaded")

dim_product.to_sql('DimProduct', engine, if_exists='append', index=False)
print("DimProduct loaded")

dim_date.to_sql('DimDate', engine, if_exists='append', index=False)
print("DimDate loaded")

Loading dimension tables...
DimCustomer loaded
DimStore loaded
DimPayment loaded
DimProduct loaded
DimDate loaded


**Load Fact Table**

The fact table is loaded after all dimension tables have been populated to ensure foreign key relationships are valid.

In [229]:
print(fact_sales.columns)

Index(['Invoice_ID', 'DateKey', 'CustomerKey', 'ProductKey', 'StoreKey',
       'PaymentKey', 'Quantity', 'Unit_price', 'Tax_5%', 'Sales', 'cogs',
       'gross_margin_percentage', 'gross_income', 'Rating'],
      dtype='str')


In [231]:
print("Loading FactSales...")

fact_sales.to_sql('FactSales', engine, if_exists='append', index=False, chunksize=1000)

print("FactSales loaded")

Loading FactSales...
FactSales loaded


**Validate Data Warehouse Tables**

After loading the tables into SQL Server, a validation step confirms that the expected number of rows exists in each table.

In [232]:
print("Connecting to SQL Server...")

conn = pyodbc.connect(
"DRIVER={ODBC Driver 17 for SQL Server};"
"SERVER=.;"
"DATABASE=RetailDW;"
"Trusted_Connection=yes;"
)

cursor = conn.cursor()

print("Connected to SQL Server Successfully!")

Connecting to SQL Server...
Connected to SQL Server Successfully!


In [233]:
print("Checking tables row counts...")

tables = [
'DimCustomer',
'DimStore',
'DimProduct',
'DimPayment',
'DimDate',
'FactSales'
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table} rows:", count)

Checking tables row counts...
DimCustomer rows: 4
DimStore rows: 3
DimProduct rows: 993
DimPayment rows: 3
DimDate rows: 89
FactSales rows: 1000


**ETL Pipeline Completed**

The ETL pipeline successfully:

- Extracted raw retail sales data
- Cleaned and transformed the dataset
- Constructed a dimensional star schema
- Coaded the data warehouse tables into SQL Server

The resulting schema can now be used for analytical queries, reporting, and business intelligence tools such as Power BI or Tableau.